# Cambodian ALPR — Vision Transformer (ViT) Experiment

## 1. Research Question & Hypothesis

This notebook trains and evaluates **Approach E: Vision Transformer (`vit_b_16`)** on the 26-class Khmer province classification task.

### Existing Baseline Approaches (from your defense study):
- **Approach A (ResNet-18 Scratch)**: 93.3% test accuracy (11.2M params)
- **Approach B (ResNet-18 Frozen)**: 42.0% test accuracy (13K trainable params — *domain shift failure*)
- **Approach C (ResNet-18 Fine-tune)**: 94.0% test accuracy (96.5% tuned)
- **Approach D (SmallCNN Scratch)**: 46.6% test accuracy (0.4M params — *capacity failure*)

### What Approach E (ViT) Tests:
1. **Self-Attention vs. Convolutions**: Does global patch attention capture Khmer orthographic ligatures better than localized 3×3 convolutions?
2. **Capacity & Representation**: ViT-B/16 has ~86M parameters. Does this larger representation improve accuracy on difficult pairs (e.g., *Banteay Meanchey* vs. *Oudor Meanchey*) or does it overfit on 2,515 images?
3. **Resolution**: ViT operates on 224×224 images (16×16 patches = 196 tokens), capturing finer stroke detail than 128×128 CNN crops.

In [ ]:
# Check available GPU accelerator (T4, V100, or A100)
!nvidia-smi -L

In [ ]:
# Mount Google Drive and unpack project bundle
from google.colab import drive
drive.mount('/content/drive')
BUNDLE = '/content/drive/MyDrive/ALPR/alpr_colab_bundle.zip'

import os, zipfile, shutil
shutil.rmtree('/content/alpr', ignore_errors=True)
os.makedirs('/content/alpr', exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall('/content/alpr')
%cd /content/alpr

In [ ]:
# Verify bundle has the updated ViT code and province crops
import glob

tr = open('scripts/recognition/train_province_classifier.py', encoding='utf-8').read()
ok_vit = 'vit_b_16' in tr
n_train = len(glob.glob('data/province_crops/train/*/*.jpg'))
n_test  = len(glob.glob('data/province_crops/test/*/*.jpg'))

print('ViT support in training script:', 'PRESENT' if ok_vit else '*** MISSING ***')
print(f'Province crops: train={n_train} | test={n_test}')
assert ok_vit, 'Bundle is missing ViT updates. Re-run make_colab_bundle.py.'
assert n_train >= 2500 and n_test >= 560, 'Province crops missing from bundle.'
print('Bundle is ready.')

In [ ]:
# Install dependencies
!pip -q install torch torchvision pyyaml tqdm pillow opencv-python-headless

In [ ]:
# Baseline test: Deployed ResNet18 on the 567 test crops
!python scripts/tools/test_province_rotation.py \
    --weights models/recognition/province_classifier_best.pth

In [ ]:
# Train Approach E: ViT-B/16 with ImageNet pre-trained weights
# Batch size 32, LR 1e-4, 40 epochs (standard training budget)
!python scripts/recognition/train_province_classifier.py \
    --arch vit_b_16 \
    --pretrained \
    --epochs 40 \
    --batch 32 \
    --lr 1e-4 \
    --out models/recognition/province_classifier_vit.pth

In [ ]:
# Evaluate ViT on the exact same 567 held-out test crops
!python scripts/tools/test_province_rotation.py \
    --weights models/recognition/province_classifier_vit.pth \
    --config models/recognition/province_classifier_vit_config.json

In [ ]:
# Save trained model & configuration to Google Drive for local evaluation
import os, shutil
os.makedirs('/content/drive/MyDrive/ALPR/trained', exist_ok=True)
saved = 0
for f in ('province_classifier_vit.pth', 'province_classifier_vit_config.json'):
    p = 'models/recognition/' + f
    if os.path.exists(p):
        shutil.copy(p, '/content/drive/MyDrive/ALPR/trained/')
        print('Saved to Drive:', f)
        saved += 1
if saved == 2:
    print('Successfully saved ViT weights and config to Google Drive.')
else:
    print('Warning: Checkpoint files missing. Review training log above.')